Systematic Model Comparison Pipeline

Compare multiple models using same CV folds with sklearn Pipeline

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
import kagglehub
from kagglehub import KaggleDatasetAdapter, dataset_load
import time

# Prepare data
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
)

# Convert 'TotalCharges' to numeric, handling missing/non-numeric values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(0, inplace=True) # Fill NaNs created by coercion with 0

target = 'Partner'
# Convert target variable to numerical (0 or 1)
y = df[target].map({'Yes': 1, 'No': 0})

# Drop 'customerID' and the original 'target' column from features
X = df.drop(columns=['customerID', target])

# Identify categorical columns for one-hot encoding
categorical_cols = X.select_dtypes(include='object').columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True) # One-hot encode categorical features

X_train , X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, stratify=y, random_state=42
)

models = {
    '1. Baseline': DummyClassifier(strategy='most_frequent'),
    '2. LogRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),
    '3. Radom Forest': RandomForestClassifier(
        n_estimators=100, random_state=42
    ),
    '4. Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, random_state=42
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results =[]

for name, model in models.items():
  start = time.time()
  scores = cross_val_score(
      model, X_train, y_train, cv=cv, scoring='accuracy'
  )
  elapsed = time.time() - start
  results.append({
      'Model:' : name,
      'CV Mean' : f"{scores.mean():.3f}",
      "CV Std" : f"{scores.std():.3f}",
      "Time" : f"{elapsed:.2f}"
  })
  print(f"{name}: {scores.mean():.3f} +/- {scores.std():.3f}")

#Evaluation
df_results = pd.DataFrame(results) # Fixed typo: pd.Dataframe -> pd.DataFrame
print("\n" + df_results.to_string(index=False))

best = models['4. Gradient Boosting']
best.fit(X_train, y_train)
print(f"\nTest Score: {best.score(X_test, y_test):.3f}")

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
1. Baseline: 0.517 +/- 0.000


/tmp/ipykernel_1423/3452578206.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(0, inplace=True) # Fill NaNs created by coercion with 0


2. LogRegression: 0.754 +/- 0.005
3. Radom Forest: 0.738 +/- 0.008
4. Gradient Boosting: 0.755 +/- 0.004

              Model: CV Mean CV Std Time
         1. Baseline   0.517  0.000 0.02
    2. LogRegression   0.754  0.005 0.24
     3. Radom Forest   0.738  0.008 5.50
4. Gradient Boosting   0.755  0.004 5.78

Test Score: 0.742


Statistical Model Comparison with Paired t-test

Check if score differences between models are statistically significant

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from scipy.stats import ttest_rel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


#get fold by fold score not just mean

# for actual use we have to use the scaler (max iter limit reach warning)
lr_pipeline = Pipeline([
    # If X_train has mixed data, use your 'preprocessor' here instead of StandardScaler
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

# 2. Pass the PIPELINE into cross_val_score, not just the model

scores_lr = cross_val_score(
    lr_pipeline, X_train, y_train, cv=cv
)

# scores_lr = cross_val_score(
#     LogisticRegression(max_iter=1000), X_train, y_train, cv=cv
# )

scores_rf = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=cv
)

#print fold by fold results
print(f" Fold: LogReg    RF")
print("-" * 25)
for i, (s1, s2) in enumerate(zip(scores_lr, scores_rf)):
  marker = "<--" if abs(s1-s2) > 0.03 else ""
  print(f"  {i+1}   {s1:.3f}   {s2:.3f}")

print(f"\nLogReg: {scores_lr.mean():.3f} +/- {scores_lr.std():.3f}")
print(f"RF:     {scores_rf.mean():.3f} +/- {scores_rf.std():.3f}")

# Pair ttest checks is the difference significant??

stat, pvalue = ttest_rel(scores_lr, scores_rf)
print(f"\nPaired t-test p-value: {pvalue:.4f}")

if pvalue < 0.05:
  better = "RF" if scores_rf.mean() > scores_lr.mean() else "LogReg"
  print(f"Significant difference (p < 0.05): {better} is better")
else:
  print("NOT significant (p >= 0.05): models are equivalent")
  print("Choose simpler model (LogReg) per Occam's Razor")

 Fold: LogReg    RF
-------------------------
  1   0.747   0.742
  2   0.762   0.752
  3   0.758   0.735
  4   0.752   0.729
  5   0.752   0.733

LogReg: 0.754 +/- 0.005
RF:     0.738 +/- 0.008

Paired t-test p-value: 0.0098
Significant difference (p < 0.05): LogReg is better


In [25]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import kagglehub
from kagglehub import KaggleDatasetAdapter, dataset_load

# ============================================================
# DEFINE PREPROCESSING (prevents data leakage)
# ============================================================
# Identify column types

# Reload data and split to get the *original* X_train and y_train for this pipeline
# This is to ensure the ColumnTransformer correctly identifies categorical and numerical columns
# from the raw, un-encoded data, preventing data leakage during preprocessing steps.
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df_pipeline = dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
)

df_pipeline['TotalCharges'] = pd.to_numeric(df_pipeline['TotalCharges'], errors='coerce')
df_pipeline['TotalCharges'] = df_pipeline['TotalCharges'].fillna(0)

target_col = 'Partner'
y_pipeline = df_pipeline[target_col].map({'Yes': 1, 'No': 0})
X_pipeline = df_pipeline.drop(columns=['customerID', target_col])

X_train_pipeline, X_test_pipeline, y_train_pipeline, y_test_pipeline = train_test_split(
    X_pipeline, y_pipeline, test_size=0.2, stratify=y_pipeline, random_state=42
)

# Now identify column types from X_train_pipeline (which still has original categorical columns)
numeric_cols = X_train_pipeline.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train_pipeline.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    [('num', StandardScaler(), numeric_cols),
     ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)],
    remainder='passthrough' # Keep any other columns if they exist (though unlikely after selection)
)

candidates = {
    'LogReg': Pipeline([
        ('prep', preprocessor),
        ('model', LogisticRegression(max_iter=1000))
    ]),
    'RF': Pipeline([
        ('prep', preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=100, random_state=42
        ))
    ])
}

# ============================================================
# EVALUATE ALL CANDIDATES ON SAME FOLDS
# ============================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Model Selection Results:")
print("=" * 40)
for name, pipeline in candidates.items():
  scores = cross_val_score(pipeline, X_train_pipeline, y_train_pipeline, cv=cv, scoring='f1')
  print(f" {name}: F1 = {scores.mean():.3f} +/- {scores.std():.3f} ")

# Pipeline ensures:
# 1. Scaler fits ONLY on training folds (no leakage)
# 2. Same preprocessing for all models (fair comparison)
# 3. One object to save/deploy (model + preprocessing)


Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Model Selection Results:
 LogReg: F1 = 0.739 +/- 0.008 
 RF: F1 = 0.710 +/- 0.011 


Complete Model Selection with sklearn Pipeline

Production-ready pipeline preventing data leakage across all candidates